## **Chemical Equilibrium: select your reaction**

<div align="left">
  <table border="1" cellpadding="6" cellspacing="0">
    <tr>
      <td bgcolor="#444444">
        <font color="#ffeb3b"><tt><b>Last updated (YYYY-MM-DD): 2026-04-10</b></tt></font>
      </td>
    </tr>
  </table>
</div>



In this notebook, you will analyze **a gas-phase elementary reaction of your choice** as a concrete case study to explore the fundamental principles governing chemical equilibrium. You will begin by specifying the reaction stoichiometry (reactants, products, and stoichiometric coefficients). The notebook then uses that input to build a unified computational workflow for predicting equilibrium behavior.

The equilibrium will be examined from two complementary perspectives: **classical thermodynamics**, which provides macroscopic equilibrium criteria through the minimization of appropriate thermodynamic potentials; and **statistical thermodynamics**, which connects equilibrium properties to molecular-level information (e.g., energies and partition functions).
<!--
and chemical kinetics, which describes the time-dependent evolution of the reacting system and its dynamic approach to equilibrium.
-->

The notebook is designed to guide you through a quantitative description of equilibrium by (i) analyzing how thermodynamic potentials vary with the extent of reaction, (ii) determining equilibrium compositions from minimization criteria under specified constraints, and (iii) exploring how changes in temperature, pressure, or initial composition shift the equilibrium position. Overall, the goal is to present equilibrium not as a static condition, but as a **predictable outcome** that can be understood consistently from thermodynamic,
and molecular viewpoints within a single computational framework.


### **Preparing the Computational Environment**  


*The next three cells prepare the environment by installing the necessary packages, importing libraries, and loading constants and functions. Make sure to run hem sequentially before doing anything else.*

⏳ **Note:** This may take **a few minutes** to complete. ⏳  

In [ ]:
#@title <small> 💻 Install Libraries (this may take a while) <small> { display-mode: "form" }
%%capture captured_output

# --- system (for LaTeX text in matplotlib) --- #
! sudo apt update -y
! sudo apt install -y cm-super dvipng texlive-latex-extra texlive-latex-recommended

# --- Python: install and update core scientific libraries --- #

# versions that are known to work well together will be selected to avoid compatibility issues
%pip install "numpy==2.0.2"
%pip install "scipy==1.16.3"

# install pyscf: as pyscf[geomopt,dispersion] gives problems, they are installed separately
%pip install --prefer-binary "pyscf==2.11.0"
%pip install "geometric==1.1"
%pip install "pyscf-dispersion==1.3.0"

%pip install "py3Dmol==2.5.3"
%pip install "pubchempy==1.0.5"
%pip install "rdkit==2025.9.1"

#!pip index versions pyscf # check available versions
# %pip install "pyberny==0.6.3"
# !pip show pyberny # check installed version

In [ ]:
#@title <small> 💻 Import Libraries and Load Constants<small> { display-mode: "form" }

# =====    Import libraries of interest    ===== #
# ---------------------------------------------- #
import os                                        #
import time                                      #
# ---------------------------------------------- #
import numpy                as np                #
#print(np.__version__)                           #
# ---------------------------------------------- #
import matplotlib.pyplot    as plt               #
import matplotlib.ticker    as mticker           #
import plotly.graph_objects as go                #
from   plotly.subplots      import make_subplots #
# ---------------------------------------------- #
from   scipy.optimize  import minimize           #
from   scipy.optimize  import minimize_scalar    #
from   scipy.optimize  import root_scalar        #
# ---------------------------------------------- #
import pubchempy       as     pcp                # to retrieve from PubChem
# ---------------------------------------------- #
import rdkit                                     # SMILES --> coordinates
from   rdkit           import RDLogger           #
from   rdkit           import Chem               #
from   rdkit.Chem      import AllChem            #
# ---------------------------------------------- #
import py3Dmol                                   # 3D visualization of molecules
# ---------------------------------------------- #
import pyscf                                     # for elec struct calculations
from   pyscf.geomopt   import geometric_solver   #
from   pyscf.hessian   import thermo             #
# ---------------------------------------------- #
import ipywidgets      as     w                  # to add buttons
# ---------------------------------------------- #
from   google.colab    import files              # to access to generated files
from   google.colab    import output             #
from   IPython.utils   import io                 # to capture output
from   IPython.display import HTML               # needed for 3D visualization
from   IPython.display import display            # needed for 3D visualization
from   IPython.display import Markdown           # needed for 3D visualization
# ============================================== #

# ---- Define some constants of interest (SI) ----
m_u    = 1.66053886e-27      # atomic mass constant in kg
m_e    = 9.1093837E-31       # mass of electron
q_e    = 1.60217663E-19      # charge of electron
h      = 6.6260693E-34       # Planck's constant in J*s
k_B    = 1.3806505E-23       # Boltzmann's constant in J/K
c_0    = 2.99792558E8        # Speed of light in m/s
eps0   = 8.85418782E-12      # Vacuum permittivity
NA     = 6.02214076E23       # Avogadro's number
# ------------------------------------------------
P_o    = 1E5                 # standard pressure (P^o = 1E5 Pa = 1 bar)
c_o    = 1E3                 # 1 mol/L = 1E3 mol/m^3
R      = NA * k_B            # gas constant J/(K mol)
# ---- Universal constants ----> Atomic Units ----
hbar    = h/(2*np.pi)
a_0     = (4*np.pi*eps0*hbar**2) / (m_e * q_e**2)
Eh      = q_e**2 / (4*np.pi * eps0 * a_0)
Hz_au   = Eh/hbar
# ------------------------------------------------
ZERO1  = 1e-300  # to avoid problems when n=0 in n*log(n)
ZERO2  = 1E-014  # if ni < ZERO2 --> ni = 0 (for chemical kinetics)
ZERO3  = 1E-010  # comparison of rotational constants (if equal --> linear)
ZERO4  = 1E-007  # for intercept method; if y=0 --> y=ZERO4 (so we can obtain the derivative)
# ------------------------------------------------
last_fig  = None
last_info = None
# ------------------------------------------------
FONTSIZE  = [11,12,14,15,16,20]
# ------------------------------------------------
NPOINTS   = 125  # number of time points
RELDIFF   = 1.00 # 1%
REL_XI_EQ = 0.98 # equil. at REL_XI_EQ * xi_eq
# ------------------------------------------------

In [ ]:
#@title <small> 💻 Load Auxiliary Functions<small> { display-mode: "form" }

#==================================#
# ---- Interaction with student ----
def ask_for_float(question,ntries=3):
    count = 0
    while True:
      try:
        value = float(input(question))
        break
      except:
        count += 1
        if count == ntries:
              print("      getting data failed too many times... aborting!")
              raise Exception
        print("      something went wrong... trying again...")
    return value

TEXT1 = '''
===================
Firstly, introduce your reaction, using the following format:

      A + 2 B -> 3 C + 2 D

Make sure to include blank spaces between the stoichiometric coefficients and the chemical species.
For example, for the reaction we studied above, you should enter:

     N2O4 -> 2 NO2
===================

'''

TEXT2 = '''
~~~~~~~~~~~~~~~~~~~
Information:

 * reactants (%i): %s
 * products  (%i): %s

 * equation for the reaction: %s
~~~~~~~~~~~~~~~~~~~

'''

# ---- Functions for getting the user reaction ----
def ask_for_reaction(max_nerr=3):

    nus0,molecules0 = None,None

    print(TEXT1)
    answer,nerr = "n",0
    while True:
        if nerr == max_nerr:
            print("-------------------")
            print("Too many errors... aborting...")
            print("-------------------")
            return nus0,molecules0
        try:
            reaction = input(" * insert reaction: ")
            print("")
            nus,molecules = string_to_reaction(reaction)
            if nus is None: raise Exception

            #---- Print info to make sure all is correct ----
            string = reaction_to_string(nus,molecules)
            nR = len([nu for nu in nus if nu<0])
            nP = len([nu for nu in nus if nu>0])
            sR = ", ".join([molecule for nu,molecule in zip(nus,molecules) if nu < 0])
            sP = ", ".join([molecule for nu,molecule in zip(nus,molecules) if nu > 0])
            print(TEXT2%(nR,sR,nP,sP,string))
            answer = input("Is the reaction correct? (Type 'yes' or 'no'): ")
            answer = answer.strip().lower()[0]
            if answer == "y":
               print("\nGreat! Now, you are ready to study your equilibrium!\n\n")
               return nus,molecules
            else:
               print("Ok, so let us try it again! :)\n\n")
               nerr += 1
        except:
            nerr += 1
            print("-------------------")
            print("There was some kind of problem... Let us try again!")
# ------------------------------------------------

#==================================#
# ---- Get reaction from the string given by user ----
def string_to_reaction(string):
    reactants, products = string.split("->")
    reactants = [i.strip() for i in reactants.split("+")]
    products  = [i.strip() for i in  products.split("+")]
    nus,molecules = [],[]
    for r in reactants:
        r = [i.strip() for i in r.split()]
        if   len(r) == 1: nu,molecule = 1,r[0]
        elif len(r) == 2: nu,molecule = r
        else            : raise Exception
        nus.append(-int(nu))
        molecules.append(molecule)
    for r in products:
        r = [i.strip() for i in r.split()]
        if   len(r) == 1: nu,molecule = 1,r[0]
        elif len(r) == 2: nu,molecule = r
        else            : raise Exception
        nus.append(+int(nu))
        molecules.append(molecule)
    return np.array(nus),np.array(molecules)
#---------------------------------------------------------------------#
def plot3D_vibdof():
    # Frequencies (1/cm)
    xx    = np.linspace(1 ,2000,201)
    # Temperatures (K)
    yy    = np.linspace(10, 500,201) # temperature
    # Contribution (0,1)
    xx,yy = np.meshgrid(xx,yy)
    zz    = vib_contribution(xx,yy)

    # Generate plot
    fig = go.Figure()
    fig.add_trace(go.Surface(x=xx,y=yy,z=zz,colorscale='Viridis',showscale=True))
    fig.update_layout(
        width=650, height=500,
        margin=dict(l=0, r=0, t=40, b=0),
        scene=dict(
            xaxis=dict(title=dict(text='freq (1/cm)' ,font=dict(size=18))),
            yaxis=dict(title=dict(text='T (K)'       ,font=dict(size=18))),
            zaxis=dict(title=dict(text='contribution',font=dict(size=18))),
            camera=dict(eye=dict(x=1.6, y=1.6, z=0.9))
        ))

    options = {"format":"svg","filename":"equilibrium_surface","scale":2}
    config  = {"toImageButtonOptions":options}
    fig.show(config=config)
#---------------------------------------------------------------------#


# ---- Write the equation of the reaction ----
def reaction_to_string(nus,molecules):
    stringR = []
    stringP = []
    for nu, molecule in zip(nus,molecules):
        if nu < 0: stringR += ["%s * %s"%(-nu,molecule)]
        if nu > 0: stringP += ["%s * %s"%(+nu,molecule)]
    return " + ".join(stringR) + " ⇌ " + " + ".join(stringP)

# ---- button to download file ----
def download_file(fname):
    btn = w.Button(
            description=f"Download {fname}",
            icon="download",
            button_style="primary",
            layout=w.Layout(width='250px', height='38px', margin='0 0 0 55px')
    )
    # action when clicking
    btn.on_click(lambda _: files.download(fname))
    return btn

def _on_download_clicked(_,_case_):
    if _case_.startswith("kin"): init_conditions = f"{T_slider.value:.0f}K_{P_slider.value:.2f}bar_{V_slider.value:.2f}L_{yA_slider.value:.2f}"

    # --- get figure from global variable ---
    fig = last_fig
    if fig is None:
        print("No figure yet; move a slider to generate the plot...")
        return

    # --- get file name ---
    if   _case_ == "PT"       : fname = f"equilibrium_T{T_slider.value:.0f}K_p{P_slider.value:.2f}bar.svg"
    elif _case_ == "VT"       : fname = f"equilibrium_T{T_slider.value:.0f}K_V{V_slider.value:.2f}L.svg"
    elif _case_ == "intercept": fname = f"intercept_T{T_slider.value:.0f}K_p{P_slider.value:.2f}bar.svg"
    elif _case_ == "dof"      : fname = "vib_dof.svg"
    elif _case_ == "kinTP"    : fname = f"chemkinTP_{init_conditions:s}.svg"
    elif _case_ == "kinTV"    : fname = f"chemkinTV_{init_conditions:s}.svg"
    else                      : raise Exception

    if _case_.startswith("kin"): original_size = fig.get_size_inches()
    if _case_.startswith("kin"): fig.set_size_inches(10,8)
    fig.savefig(fname, bbox_inches='tight',dpi=300)
    if _case_.startswith("kin"): fig.set_size_inches(original_size)
    files.download(fname)
#=====================================================

#=====================================================
# FOR CHEMICAL THERMODYNAMICS
#=====================================================

# ============================================== #
# For all the following functions, it is assumed #
# that all data are provided in SI units.        #
# Moreover, P is used for total pressure,        #
# whereas lower p is used for partial pressures. #
# ============================================== #

# ---- Limits for extent of reaction ----
def limits_xi(n_0,nus):

    # maximum value of xi (calculated considering consumption of reactants)
    xi_max = min([-n_0_i/nu_i for n_0_i,nu_i in zip(n_0,nus) if nu_i < 0])

    # minimum value of xi (calculated considering consumption of products)
    xi_min = max([-n_0_i/nu_i for n_0_i,nu_i in zip(n_0,nus) if nu_i > 0])

    # Ensure numerical zeros are displayed as +0.0 for clarity
    if xi_min == -0.0: xi_min = 0.0
    if xi_max == -0.0: xi_max = 0.0

    return xi_min,xi_max
#-----------------------------------------------------

# ---- auxiliary function ----
def prepare_variables(T, P, nus, n_0, xi):

    T   = np.asarray(T)
    P   = np.asarray(P)
    nus = np.asarray(nus) #, dtype=float)  # (S,)
    n_0 = np.asarray(n_0) #, dtype=float)  # (S,)
    xi  = np.asarray(xi)

    # Expand for broadcasting with xi (which can be a scalar or have shape (M, N))
    # If xi.ndim == 0 (scalar), this keeps the shape (S,) -> (S,)
    # If xi.ndim == 2, it becomes (S, 1, 1) and broadcasts to (S, M, N)
    expand = (None,)*xi.ndim
    n_0_e  = n_0[(...,) + expand]
    nus_e  = nus[(...,) + expand]

    # total sums (scalars)
    dnu    = nus.sum()
    ntot_0 = n_0.sum()

    return T, P, nus_e, n_0_e, xi, dnu, ntot_0

# ---- Delta{H}^o as a function of T ----
def get_DHo(T,refdata):
    DHo_ref,DSo_ref,DCPo_ref,T_ref = refdata
    DHo_T = DHo_ref + DCPo_ref * T * (1 - T_ref/T)
    return DHo_T

# ---- Delta{G}^o as a function of T ----
def get_DGo(T,refdata):
    DHo_ref,DSo_ref,DCPo_ref,T_ref = refdata
    DGo_T  = DHo_ref - T * DSo_ref
    DGo_T += DCPo_ref  * ( T - T_ref + T*np.log(T_ref / T))
    return DGo_T

# ---- DeltaG* ----
def get_Gast(T,P,nus,refdata):
    DGast = get_DGo(T,refdata) + R*T*np.log(P/P_o) * nus.sum()
    return DGast

# ---- DeltaGmix ----
def get_DDGmix(xi, T, P, n_0, nus):
    """
    Accepts xi as either a scalar or an array (e.g., a mesh of shape (M, N)); T and p can be scalars or broadcast-compatible arrays.
    """

    T, P, nus, n_0, xi, dnu, ntot_0 = prepare_variables(T, P, nus, n_0, xi)

    # magnitudes dependent on xi
    n_xi    = n_0    + xi * nus   # (S,) o (S,M,N)
    ntot_xi = ntot_0 + xi * dnu   # scalar or (M,N)

    # Get molar fractions
    y_xi = n_xi /ntot_xi
    y_0  = n_0  /ntot_0

    # Notice that 0*ln(0) --> 0
    maskA = (n_xi > ZERO1) & (y_xi > ZERO1)
    termA = np.zeros_like(y_xi, dtype=float)
    termA[maskA] = n_xi[maskA] * np.log(y_xi[maskA])

    maskB = (n_0  > ZERO1) & (y_0  > ZERO1)
    termB = np.zeros_like(y_0, dtype=float)
    termB[maskB] = n_0[maskB]  * np.log(y_0[maskB])

    DDGmix = R*T*np.sum(termA-termB, axis=0)
    return DDGmix

#==========================================#
# ----           Constant T,P           ----
#==========================================#

# ---- Gibbs free energy for a given T,p ----
def get_G_TP(xi,n_0,nus,P,T,refdata):
    Gast   = get_Gast(T,P,nus,refdata)
    DDGmix = get_DDGmix(xi,T,P,n_0,nus)
    DGtot  = Gast * xi + DDGmix
    return DGtot

# ---- Quotient of reaction (pressure) ----
def get_Qp_TP(n_0,nus,xi,P):

    # Get partial pressures for this value of xi
    n_xi = n_0 + xi*nus
    y_xi = n_xi/n_xi.sum()
    p_xi = y_xi * P
    with np.errstate(divide='ignore', invalid='ignore', over='ignore', under='ignore'):
         Qp = np.prod([(p_xi_j/P_o)**nu_j for nu_j,p_xi_j in zip(nus,p_xi)])
    return Qp

# ---- Extent of reaction in terms of P and T ----
def get_xieq_TP(P,T,n_0,nus,refdata):

    # Get Eq constant
    dGo_T = get_DGo(T,refdata)
    Kp_T  = np.exp(-dGo_T/R/T)

    # Get limits for xi
    xi_min,xi_max = limits_xi(n_0,nus)
    xi_guess      = 0.5 * (xi_max + xi_min)
    result        = root_scalar(lambda xi: get_Qp_TP(n_0,nus,xi,P)-Kp_T,x0=xi_guess,bracket=(xi_min,xi_max),xtol=1E-8)
    xi_eq         = float(result.root)
    return xi_eq

#==========================================#
# ----           Constant T,V           ----
#==========================================#

# ---- Helmholtz free energy for a given T,V ----
def get_A_TV(xis,T,V,n_0,nus,refdata):

    ntot_0  = n_0.sum()
    ntot_xi = ntot_0 + nus.sum() * xis
    DGo_T   = get_DGo(T,refdata)

    P_xi    = ntot_xi*R*T/V

    term_lineal = DGo_T * xis
    termA   = ntot_xi * R *T * (np.log( P_xi/P_o                  )-1)
    termA  -= ntot_0  * R *T * (np.log((P_xi/P_o)*(ntot_0/ntot_xi))-1)
    DDGmix  = get_DDGmix(xis,T,P_xi,n_0,nus)
    term_nolin  = termA+DDGmix

    DAtot   = term_lineal + term_nolin

    return DAtot, term_lineal, term_nolin, P_xi

# ---- Quotient of reaction (volume) ----
def get_Qp_TV(n_0,nus,xi,V,T):

    # Get partial pressures for this value of xi

    n_xi = n_0 + xi*nus
    y_xi = n_xi/(n_xi.sum())
    P    = n_xi.sum() *R*T/V
    p_xi = y_xi * P
    with np.errstate(divide='ignore', invalid='ignore', over='ignore', under='ignore'):
         Qp = np.prod([(p_xi_j/P_o)**nu_j for nu_j,p_xi_j in zip(nus,p_xi)])
    return Qp

# ---- Extent of reaction in terms of V and T ----
def get_xieq_TV(V,T,n_0,nus,refdata):

    # Get Eq constant
    dGo_T = get_DGo(T,refdata)
    Kp_T  = np.exp(-dGo_T/R/T)

    # Get limits for xi
    xi_min,xi_max = limits_xi(n_0,nus)
    xi_guess      = 0.5 * (xi_max + xi_min)
    result        = root_scalar(lambda xi: get_Qp_TV(n_0,nus,xi,V,T)-Kp_T,x0=xi_guess,bracket=(xi_min,xi_max))
    xi_eq         = float(result.root)
    return xi_eq

#==================================#
# ---- FUNCTIONS FOR PLOTTING ---- #
#==================================#
def plot_DG_T(T,T_ref,refdata):
    # Calculation of DG at Tref
    DGo_ref = get_DGo(T_ref,refdata)
    Keq_ref = np.exp(-DGo_ref/R/T_ref)

    # Calculation of DG at other temperatures
    DGo_T = get_DGo(T,refdata)
    Keq_T = np.exp(-DGo_T/R/T)

    # Plot results
    plt.rcParams['text.usetex'] = True

    # Create figure with two panels side by side
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))

    # LEFT PANEL #
    labeldot = r"$\Delta_{r}{G}^\circ(T_{\rm ref}) = %.2f \;\; \mathrm{kJ/mol}$"%(DGo_ref/1000)
    ax[0].plot(T    ,DGo_T  /1000,'k-',zorder=1)
    ax[0].plot(T_ref,DGo_ref/1000,'ro',zorder=3,label=labeldot)
    ax[0].tick_params(axis='both', labelsize=FONTSIZE[4])
    ax[0].set_xlabel(r'$T \;\; (\mathrm{K})$'                        ,fontsize=FONTSIZE[5])
    ax[0].set_ylabel(r'$\Delta_{r} G^{\circ} \;\; (\mathrm{kJ/mol})$',fontsize=FONTSIZE[5])
    ax[0].legend(loc="best",fontsize=FONTSIZE[3])

    # RIGHT PANEL #
    if 1E-2 < Keq_ref < 1000: sKeq_ref = "%.3f"%Keq_ref
    else                    : sKeq_ref = "%.3e"%Keq_ref
    labeldot = r"$K_{p}^\circ(T_{\rm ref}) = %s$"%sKeq_ref
    ax[1].plot(T    ,Keq_T  ,'k-',zorder=1)
    ax[1].plot(T_ref,Keq_ref,'ro',zorder=3,label=labeldot)
    ax[1].tick_params(axis='both', labelsize=FONTSIZE[4])
    ax[1].set_xlabel(r'$T \;\; (\mathrm{K})$'     ,fontsize=FONTSIZE[5])
    ax[1].set_ylabel(r'$K_{p}^\circ(T_{\rm ref})$',fontsize=FONTSIZE[5])
    ax[1].legend(loc="best",fontsize=FONTSIZE[3])

    # ---- download button ----
    fname = 'plot_DGo_T.svg'
    fig   = plt.gcf()
    fig.savefig(fname, bbox_inches='tight')
    btn   = download_file(fname)

    # ---- show plot and close ----
    display(btn)
    plt.show()
    plt.close()

    return DGo_T

# ---- plot DGo with temperature ----
def plot_DG_T_2(T,DGo_T,DGo_model,DCP,T_ref,DGo_ref,key):

    plt.rcParams['text.usetex'] = True

    if DCP is not None: DCP = DCP/R
    # Plot results
    plt.plot(T,[ii/1000 for ii in DGo_T]    ,'k-',zorder=1,label=r"Stat-Mech")

    if DGo_model is not None:
       plt.plot(T,[ii/1000 for ii in DGo_model],'r--',zorder=1,label=r"Eq. (1) with $\Delta_{r}C_{p}^\circ= %.2f\cdot R$"%DCP)

    if T_ref is not None:
       plt.plot(T_ref,DGo_ref/1000,'ro')

    # Format plot
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)

    plt.xlabel(r'$T \;\; (\mathrm{K})$'                        ,fontsize=16)
    plt.ylabel(r'$\Delta_{r} G^{\circ} \;\; (\mathrm{kJ/mol})$',fontsize=16)

    plt.legend(loc="best",fontsize=12)

    if DGo_model is not None:
       level = level_to_string(key[0],key[1])
       fname = rf"plot_DGo_T_stat_{level:s}_{DCP:5.2f}.svg"
       # ---- download button ----
       fig = plt.gcf()
       fig.savefig(fname, bbox_inches='tight')
       btn = download_file(fname)
       display(btn)

    # ---- show plot and close ----
    plt.show()
    plt.close()
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~#
#----------------------------------#
def plot_gibbshelmholtz(T,DGo_T,refdata):
    # --- DH^o at each temperature ---
    DHo_T = get_DHo(T,refdata)
    yy    = DHo_T / (R * T**2)

    # Plot results
    fig, ax = plt.subplots()
    ax.plot(T,yy,'k-',zorder=1,label=r'$(a) \;\; f=\Delta_r H^{\circ}/T^2$ \quad\quad\quad\quad [analytic]')

    # Format plot
    ax.tick_params(axis='x', labelsize=FONTSIZE[4])
    ax.tick_params(axis='y', labelsize=FONTSIZE[4])

    ax.set_xlabel(r'$T \;\; (\mathrm{K})$'       , fontsize=FONTSIZE[5])
    ax.set_ylabel(r'$f/R \;\; (\mathrm{K}^{-1})$', fontsize=FONTSIZE[5])

    # --- Calculate numerical derivative ---
    numslope = np.gradient(-DGo_T / (R*T) , T)

    # --- Add data to plot (removing first and last points, due to numerical error) ---
    ax.plot(T[1:-1],numslope[1:-1],'xr',zorder=2, markeredgewidth=2.5,label=r'$(b)  \;\; f = -d/dT(\Delta_r G^{\circ}/T)$ \;\, [numeric]')

    ax.legend(loc="best",fontsize=FONTSIZE[3])

    # ---- download button ----
    fname = "plot_gibbshelmholtz.svg"
    fig   = plt.gcf()
    fig.savefig(fname, bbox_inches='tight')
    btn   = download_file(fname)

    # Show plot, button and close
    display(btn)
    display(fig)
    plt.close()
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~#
#----------------------------------#
def print_info_eq(magnitude,PVT_0,PVT_eq,molecules,xi_eq,n_eq,y_eq,p_eq,P_eq,Kp_v1,Kp_v2,Ky_v1,Ky_v2):

    P_0 ,V_0 ,T_0  = PVT_0
    P_eq,V_eq,T_eq = PVT_eq

    sP_0  = rf"{P_0/1E5:.2f}"
    sP_eq = rf"{P_eq/1E5:.2f}"
    Pformat = max(len(sP_0),len(sP_eq))

    sV_0  = rf"{V_0*1E3:.2f}"
    sV_eq = rf"{V_eq*1E3:.2f}"
    Vformat = max(len(sV_0),len(sV_eq))

    print("")
    print(fr"Initial  conditions: ({T_0:.2f}K,{sP_0:{Pformat:d}s}bar,{sV_0:{Vformat:d}s}L)")
    print("")
    print(fr"Equilib. conditions: ({T_eq:.2f}K,{sP_eq:{Pformat:d}s}bar,{sV_eq:{Vformat:d}s}L)")
    print("")
    print(fr"   ==> equilibrium found at xi_eq = {xi_eq:.4f} mol")
    print("")

    print(fr"   Number of moles & molar fraction at equilibrium (from xi_eq):")
    for j,molecule in enumerate(molecules):
        n_eq_j = n_eq[j]
        y_eq_j = y_eq[j]
        print(fr"   * n_i = {n_eq_j:6.4f} mol, y_i = {y_eq_j:7.4f} (i = {molecule:s})")
    print("")

    print(fr"   Partial pressures at equilibrium:")
    for j,molecule in enumerate(molecules):
        p_eq_j = p_eq[j]/1E5
        print(fr"   * p_i = {p_eq_j:7.4f} bar (i = {molecule:s})")
    print("")


    if 9999 > Kp_v1 > 0.100: sformat1 = ".3f"
    else                   : sformat1 = ".2E"
    if 9999 > Ky_v1 > 0.100: sformat2 = ".3f"
    else                   : sformat2 = ".2E"

    reldiff = 100*abs(Kp_v2-Kp_v1)/Kp_v1
    if reldiff < 5.0:
        print(fr"   Value of Kp:")
        print(fr"   * from Delta_r{{G}}^o --> Kp = {Kp_v2:{sformat1}} [*]")
        print(fr"   * from p_i values   --> Kp = {Kp_v1:{sformat1}}")
        print("")
        print(fr"   Value of Ky:")
        if Ky_v2 is not None:
          print(fr"   * from Delta_r{{G}}^* --> Ky = {Ky_v2:{sformat2}} [*]")
        print(fr"   * from y_i values   --> Ky = {Ky_v1:{sformat2}}")
        print("")
        print(fr"   The previous values for the equilibrium constants may vary slightly")
        print(fr"   due to numerical errors. Trust the values with the [*].")
    else:
        print(fr"   Value of Kp:")
        print(fr"   * from Delta_r{{G}}^o --> Kp = {Kp_v2:{sformat1}}")
        print("")
        if Ky_v2 is not None:
          print(fr"   Value of Ky:")
          print(fr"   * from Delta_r{{G}}^* --> Ky = {Ky_v2:{sformat2}}")
    print("")
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~#
#----------------------------------#
def plot_DG_TP(T,P, fixed_args):

    molecules,nus,n_0,xis,refdata = fixed_args

    V_0 = n_0.sum()*R*T/P

    # ---- Calculate DG* ----
    DGo   = get_DGo(T,refdata)
    Gast  = get_Gast(T,P,nus,refdata)

    # ---- Terms in DGtot ----
    DGlin  = Gast * xis
    DDGmix = get_DDGmix(xis,T,P,n_0,nus)
    DGtot  = DGlin + DDGmix

    # ---- minimum of DGtot ---
    xi_eq  = get_xieq_TP(P,T,n_0,nus,refdata)
    minDG  = get_G_TP(xi_eq,n_0,nus,P,T,refdata)
    minDG  = minDG/(R*T)

    # ---- minimum of DGmix ---
    min2xi  = xis[np.argmin(DDGmix)]
    min2DG  = np.min(DDGmix)/(R*T)

    # ---- Number of moles and pressure at equilibrium ----
    n_eq = n_0 + xi_eq * nus
    y_eq = n_eq / n_eq.sum()
    p_eq = P * y_eq
    V_eq = (n_eq.sum())*R*T/P

    # ---- Calculation of Kp and Ky ----
    # (a) using molar fractions and partial pressures (from xi_eq)
    Kp_v1, Ky_v1 = 1.0, 1.0
    for p_eq_j,nu_j in zip(p_eq,nus): Kp_v1 *= (p_eq_j/P_o)**nu_j
    for y_eq_j,nu_j in zip(y_eq,nus): Ky_v1 *= y_eq_j**nu_j
    # (b) using deltaG values
    Ky_v2 = np.exp(-Gast/(R*T))
    Kp_v2 = np.exp(-DGo /(R*T))
    # (c) directly from xi_eq (specific case A --> 2B)
    Ky_v3 = 4*xi_eq * xi_eq / (1-xi_eq*xi_eq)

    # --- Plot data ---
    fig = plt.figure(figsize=(5.50,3.67))
    plt.plot(xis, DGlin/(R*T), color='b',ls="--",label=r"$f = \xi \cdot \Delta_{r} G^{\ast}$")
    plt.plot(xis,DDGmix/(R*T), color='r',ls=":" ,label=r"$f = \Delta_{\rm mix}G(\xi) - \Delta_{\rm mix}G(0)$")
    plt.plot(xis, DGtot/(R*T), color='k',ls="-" ,label=r"$f = G(\xi) - G(0)$")
    plt.plot( xi_eq ,minDG ,'ko' )
    plt.plot( min2xi,min2DG,'rx' )

    # some formatting
    plt.xticks(fontsize=FONTSIZE[1])
    plt.yticks(fontsize=FONTSIZE[1])
    plt.xlabel(r'$\xi \;\; \mathrm{[mol]}$'           , fontsize=FONTSIZE[2])
    plt.ylabel(r'$f \; / \; (RT) \;\; \mathrm{[mol]}$', fontsize=FONTSIZE[2])
    plt.title(fr'T={T:.0f} K, p={P/1E5:.2f} bar')

    # secondary grid in x-axis
    ax = plt.gca()
    ax.xaxis.set_minor_locator(mticker.MultipleLocator(0.1))
    ax.grid(True, which='minor', axis='x', alpha=0.15)
    ax.tick_params(axis='x', which='minor',bottom=False,top=False,length=0)
    ax.xaxis.set_minor_formatter(mticker.NullFormatter())

    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.legend(loc="best", fontsize=FONTSIZE[0])

    # --- update global variable: last_fig ---
    global last_fig
    last_fig = plt.gcf()

    # --- Show and close figure ---
    plt.show()
    plt.close()

    # ---- Print info ----
    PVT_0  = (P,V_0 ,T)
    PVT_eq = (P,V_eq,T)
    # print(fr"Value for Delta_r{{G}}^o({T:.2f} K) = {DGo/1000:.2f} kJ/mol")
    # print(fr"Value for Delta_r{{G}}^*({T:.2f} K) = {Gast/1000:.2f} kJ/mol")
    print_info_eq("G",PVT_0,PVT_eq,molecules,xi_eq,n_eq,y_eq,p_eq,P,Kp_v1,Kp_v2,Ky_v1,Ky_v2)
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~#
#----------------------------------#
def plot_DA_TV(T,V, fixed_args):

    molecules,nus,n_0,xis,refdata = fixed_args

    # ---- Terms in DAtot ----
    DAtot, term_lineal, term_nolin, P_xi = get_A_TV(xis,T,V,n_0,nus,refdata)

    # ---- minimum of DAtot ---
    idx_eq = np.argmin(DAtot)
    xi_eq  = xis[idx_eq]
    P_eq   = P_xi[idx_eq]
    minDA  = np.min(DAtot)/(R*T)

    # ---- minimum of DAtot ---
    xi_eq  = get_xieq_TV(V,T,n_0,nus,refdata)
    minDA  = get_A_TV(xi_eq,T,V,n_0,nus,refdata)[0]/(R*T)

    # ---- minimum of non-lineal term ---
    min2xi  = xis[np.argmin(term_nolin)]
    min2yy  = np.min(term_nolin)/(R*T)

    # ---- Number of moles and pressure at equilibrium ----
    n_eq = n_0 + xi_eq * nus
    y_eq = n_eq / n_eq.sum()
    p_eq = P_eq * y_eq

    # ---- Calculation of Kp and Ky ----
    # (a) using molar fractions and partial pressures (from xi_eq)
    Kp_v1, Ky_v1 = 1.0, 1.0
    for p_eq_j,nu_j in zip(p_eq,nus): Kp_v1 *= (p_eq_j/P_o)**nu_j
    for y_eq_j,nu_j in zip(y_eq,nus): Ky_v1 *= y_eq_j**nu_j
    # (b) using deltaG values
    DGo          = get_DGo(T,refdata)
    Kp_v2, Ky_v2 = np.exp(-DGo /(R*T)), None

    # --- Plot data ---
    plt.figure(figsize=(5.50,3.67))

    plt.plot(xis,term_lineal/(R*T), color='b',ls="--",label=r"$f = \xi \cdot \Delta_{r} G^{o}$")
    plt.plot(xis,term_nolin /(R*T), color='r',ls=":",label=r"$f = V \sum_i \left[ p_i \ln\frac{p_i}{e p^\circ} - p_i(0) \ln \frac{p_i(0)}{e p^\circ} \right]$")
    plt.plot(xis,      DAtot/(R*T), color='k',ls="-" ,label=r"$f = A(\xi) - A(0)$")

    plt.plot( xi_eq ,minDA ,'ko' )
    plt.plot(min2xi ,min2yy ,'rx' )

    plt.xticks(fontsize=FONTSIZE[1])
    plt.yticks(fontsize=FONTSIZE[1])
    plt.xlabel(r'$\xi \;\; \mathrm{[mol]}$'           , fontsize=FONTSIZE[2])
    plt.ylabel(r'$f \; / \; (RT) \;\; \mathrm{[mol]}$', fontsize=FONTSIZE[2])
    plt.title(fr'T={T:.0f} K,  V={V*1000:.2f} L')

    # secondary grid in x-axis
    ax = plt.gca()
    ax.xaxis.set_minor_locator(mticker.MultipleLocator(0.1))
    ax.grid(True, which='minor', axis='x', alpha=0.15)
    ax.tick_params(axis='x', which='minor',bottom=False,top=False,length=0)
    ax.xaxis.set_minor_formatter(mticker.NullFormatter())

    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.legend(loc="best",fontsize=FONTSIZE[0])

    # --- update global variable: last_fig ---
    global last_fig
    last_fig = plt.gcf()

    # --- Show and close figure ---
    plt.show()
    plt.close()

    # ---- Print info ----
    P_0 = n_0.sum() * R*T/V
    PVT_0  = (P_0 ,V,T)
    PVT_eq = (P_eq,V,T)
    print_info_eq("A",PVT_0,PVT_eq,molecules,xi_eq,n_eq,y_eq,p_eq,P_eq,Kp_v1,Kp_v2,Ky_v1,Ky_v2)
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~#

#=====================================================
# FOR Statistical-Mechanical
#=====================================================

def level_to_string(functional,basis):
    level      = rf"{functional.upper():s}_{basis.upper():s}"
    # replace * by _ast_ to avoid name problems
    level      = level.replace("**","_ast_ast_")
    level      = level.replace("*","_ast_")
    return level

# ---- names of files of interets ----
def files_of_interest(molecule,functional="",basis=""):

    level      = level_to_string(functional,basis)
    # filenames
    try   : sgrid = rf".grid{DFTGRID:d}"
    except: sgrid = rf""
    xyz_guess  = rf"xyz_guess-{molecule:s}.xyz"
    xyz_opt    = rf"xyz_optim-{molecule:s}.{level:s}{sgrid:s}.xyz"
    output_opt = rf"pyscf_opt-{molecule:s}.{level:s}{sgrid:s}.out"
    output_frq = rf"pyscf_frq-{molecule:s}.{level:s}{sgrid:s}.out"
    # return filenames
    return xyz_guess,xyz_opt,output_opt,output_frq

# ---- PUBCHEM: get geom ----
def pubchem_cid(cid):
    symbols,coords,smiles = None,None,None

    try   : compound = pcp.Compound.from_cid(int(cid))
    except: compound = None

    if compound is not None:
       print(rf"     - geometry retrieved!")
       print(rf"")
       print(rf"     - information:")
       print(rf"       * PubChem CID       : {compound.cid}")
       print(rf"       * molecular formula : {compound.molecular_formula:s}")
       # print(rf"       * charge            : {compound.charge:d}")
       print(rf"       * SMILES            : '{compound.smiles:s}'")
       print("")
       smiles  = compound.smiles
       # for diatomic molecules, 3D geoemtry is not stored in PubChem
       try   : geom = pcp.get_compounds(int(cid),"cid",record_type='3d')[0]
       except: geom = pcp.get_compounds(int(cid),"cid")[0]
       symbols = [atom.element           for atom in geom.atoms]
       # z-coordinate may be None (when record_type='3d' is not used)
       coords  = [tuple(0.0 if c is None else c for c in (atom.x, atom.y, atom.z)) for atom in geom.atoms]
    return symbols,coords,smiles

# ---- RDKIT: get geom (from SMILES) ----
def rdkit_smiles2geom(smiles):

    symbols,coords = None,None
    try:
       m   = Chem.AddHs(Chem.MolFromSmiles(smiles))
       cid = AllChem.EmbedMolecule(m)
       if cid >= 0:
          symbols = [atom.GetSymbol() for atom in m.GetAtoms()]
          coords  = [list(m.GetConformer().GetAtomPosition(i)) for i in range(m.GetNumAtoms())]
          mformu  = {s:0 for s in symbols}
          for s in symbols: mformu[s] += 1
          mformu  = "".join([k+str(v) if v!=1 else k for k,v in mformu.items()])
          print(rf"     - geometry generated!")
          print(rf"")
          print(rf"     - information:")
          print(rf"       * molecular formula = {mformu:s}")
          print(rf"       * SMILES            = '{smiles:s}'")
          print("")
    except: pass

    return symbols,coords,smiles

# ---- data from smiles to .xyz file ----
def data_2_xyz(symbols,xcc,fname,smiles=""):
    string  = rf"{len(symbols):.0f}"+"\n"
    string += rf"Cartesian coordinates for SMILES: {smiles:s}"+"\n"
    for idx,symbol in enumerate(symbols):
        xx,yy,zz = [coord for coord in xcc[idx]]
        string += rf"{symbol:2s}     {xx:9.5f}  {yy:9.5f}  {zz:9.5f}" + "\n"
    with open(fname,'w') as asdf: asdf.write(string)
    # download file button
    btn = download_file(fname)
    display(btn)
    print("")

# ---- read .xyz file ----
def read_xyz(filename):
    with open(filename) as f: lines = f.readlines()
    nat = int(lines[0])
    symbols = []
    coords = []
    for line in lines[2:2+nat]:
        parts = line.split()
        if len(parts) < 4: continue
        sym, x, y, z = parts[:4]
        symbols.append(sym)
        coords.append([float(x), float(y), float(z)])
    return symbols, np.array(coords)

# ---- to visualize geom in .xyz file ----
def create_visualization_xyz(xyz_file):
    # draw molecule
    view = py3Dmol.view(width=300, height=200)
    view.addModel(open(xyz_file, 'r').read(), 'xyz')
    view.setStyle({'stick': {'singleBonds': False}, 'sphere': {'scale': 0.3}})
    # Automatic labels: 0-based
    index_style = {"fontColor": "black","fontSize": 12,"showBackground": True ,"backgroundColor": "white","backgroundOpacity": 0.7}
    #elemn_style = {"fontColor": "black","fontSize": 12,"showBackground": False,"inFront": True,"alignment": "center","offset": {"x": 0, "y": 10, "z": 0}}
    view.addPropertyLabels("index",{},index_style)
    #view.addPropertyLabels("elem" ,{},elemn_style)
    #zoom
    view.zoomTo()
    view.zoom(2.5)
    return view

def geometric_info_xyz(xyz_file,geominfo):
    info = ""
    symbols, coords = read_xyz(xyz_file)
    for ii in geominfo:
        if len(ii) == 2:
          at1,at2 = ii
          distance = np.linalg.norm(coords[at1] - coords[at2])
          sbond    = rf"{symbols[at1]:s}-{symbols[at2]:s}"
          info    += rf"       * d({sbond:5s}) = {distance:.3f} Å" + "\n"
        elif len(ii) == 3:
          at1,at2,at3 = ii
          v1 = coords[at1] - coords[at2]
          v2 = coords[at3] - coords[at2]
          cosang = np.dot(v1, v2) / (np.linalg.norm(v1)*np.linalg.norm(v2))
          cosang = np.clip(cosang, -1.0, 1.0)
          angle  = np.degrees(np.arccos(cosang))
          sangle = rf"{symbols[at1]:s}-{symbols[at2]:s}-{symbols[at3]:s}"
          info  += rf"       * ∠({sangle:5s}) = {angle:.1f}°" + "\n"
    return info

# ---- geometry optization with PySCF ----
def pyscf_carryout_opt(molecule,unpaired,charge,functional,basis,bsym=False):

    # Files of interest
    xyz_guess,xyz_opt,output_opt,output_frq = files_of_interest(molecule,functional,basis)

    # Generate molecule from .xyz file
    with io.capture_output() as captured:
        mol = pyscf.gto.Mole()
        mol.atom         = xyz_guess
        mol.spin         = unpaired
        mol.charge       = charge
        mol.basis        = basis
        mol.output       = output_opt
        mol.verbose      = 4
        if bsym:
           mol.symmetry     = True
           mol.symmetrize   = True
           mol.symmetry_tol = 1e-2
        mol.build()

    # Define DFT method and mesh grid (from 0 to 9, default = 3)
    if unpaired == 0: mf = mol.RKS(xc=functional)
    else            : mf = mol.UKS(xc=functional)
    mf.grids.level = DFTGRID
    mf.max_cycle   = 200
    mf.conv_tol    = 1e-7

    # run SCF and avoid printing information
    print("     - single point calculation of guess geometry...")
    t1   = time.time()
    with io.capture_output() as captured: mf   = mf.run()
    Etot = mf.e_tot
    t2   = time.time()
    print(rf"       Etot(guess geometry) = {Etot:.5f} hartree")
    print(rf"       SCF took {t2-t1:.1f} seconds")

    # optimization [avoid printing information]
    t1   = time.time()
    print("     - geometry optimization...")
    conv_params = {}
    conv_params['convergence_energy'] = 5.0e-7  # Eh
    conv_params['convergence_gmax'  ] = 2.0e-4  # Eh/Bohr
    with io.capture_output() as captured:
       try:
          opt_geom = geometric_solver.optimize(mf, maxsteps=300, **conv_params)
       except:
          conv_params['convergence_energy'] = 1.0e-6  # Eh
          conv_params['convergence_gmax'  ] = 1.0e-4  # Eh/Bohr
          opt_geom = geometric_solver.optimize(mf, maxsteps=300, **conv_params)
    for line in opt_geom.tostring().split("\n"): print(7*" "+line)
    t2   = time.time()
    print(rf"       geometry opt. took {t2-t1:.1f} seconds")

    pyscf.gto.tofile(opt_geom,xyz_opt)

# ---- frequency calculation with PySCF ----
def pyscf_carryout_frq(molecule,unpaired,charge,functional,basis,bsym=False):

    # Files of interest
    xyz_guess,xyz_opt,output_opt,output_frq = files_of_interest(molecule,functional,basis)

    # Generate molecule from optimized geometry stored in xyz file
    with io.capture_output() as captured:
        mol = pyscf.gto.Mole()
        mol.atom         = xyz_opt
        mol.spin         = unpaired
        mol.charge       = charge
        mol.basis        = basis
        mol.output       = output_frq
        mol.verbose      = 6
        if bsym:
           mol.symmetry     = True
           mol.symmetrize   = True
           mol.symmetry_tol = 1e-2
        mol.build()

    # Define DFT method and mesh grid
    if unpaired == 0: mf = mol.RKS(xc=functional)
    else            : mf = mol.UKS(xc=functional)
    mf.grids.level = DFTGRID
    mf.max_cycle   = 200
    mf.conv_tol    = 1e-7

    print("     - Hessian calculation...")
    t1   = time.time()

    # run SCF and avoid printing information
    with io.capture_output() as captured: mf   = mf.run()
    Etot = mf.e_tot
    print(rf"       Etot(optim geometry) = {Etot:.5f} hartree")

    # Carry out Hessian calculation
    with io.capture_output() as captured: hessian = mf.Hessian().kernel()

    # add Hessian matrix to output_frq
    H4 = np.asarray(hessian)
    with open(output_frq, "a") as f:
        f.write("\n*** HESSIAN BY 3x3 ATOMIC BLOCKS (Hartree/Bohr^2) ***\n")
        for i in range(mol.natm):
            for j in range(mol.natm):
                f.write(f"\n# Block ({i+1},{j+1})  [atom {i+1} vs atom {j+1}]\n")
                np.savetxt(f, H4[i, j], fmt=" % .6e")

    t2   = time.time()
    print(rf"       Hessian calc. took {t2-t1:.1f} seconds")

    # return data
    return mol, mf, hessian

# ---- download PySCF files ----
def pyscf_download(molecule,functional,basis,which_ones=[]):

    xyz_guess,xyz_opt,output_opt,output_frq = files_of_interest(molecule,functional,basis)
    print(rf"     - file(s) to download:")
    if 1 in which_ones:
         print(rf"       {xyz_opt:s}")
         btn1 = download_file(xyz_opt)   ; display(btn1)
    if 2 in which_ones:
         print(rf"       {output_opt:s}")
         btn2 = download_file(output_opt); display(btn2)
    if 3 in which_ones:
         print(rf"       {output_frq:s}")
         btn3 = download_file(output_frq); display(btn3)

# ---- information of interest after Hessian calc ----
def pyscf_extract(mol, mf, hessian, unpaired):

    # translational info
    masses  = mol.atom_mass_list(isotope_avg=True)
    mass    = sum(masses)

    # vibrational info
    freqs       = thermo.harmonic_analysis(mf.mol, hessian)['freq_au']
    au2hz       = (1/2/np.pi)*(Eh/m_u/a_0**2)**0.5
    freqs_Hz    = [f*au2hz for f in freqs]
    wavenum_m   = [f/c_0   for f in freqs_Hz]

    info_thermo = thermo.thermo(mf, freqs) # by default, at 298.15 K and 101325 Pa
    ZPE         = info_thermo['ZPE'][0]

    # rotational info
    A,B,C = thermo.rotation_const(masses,mol.atom_coords(),unit='GHz')
    # if linear, two are equal, the other is infinity
    if   np.isinf(A) and abs(B-C) < ZERO3: A,B,C,linear = B,None,None,True
    elif np.isinf(B) and abs(A-C) < ZERO3: A,B,C,linear = C,None,None,True
    elif np.isinf(C) and abs(A-B) < ZERO3: A,B,C,linear = A,None,None,True
    else                                 :       linear =             False
    sigma = thermo.rotational_symmetry_number(mol)

    # electronic info
    E0 = info_thermo['E0' ][0]

    # collect data
    data = {}
    data["natoms"  ] = mol.natm
    data["mass"    ] = mass
    data["rotcons" ] = [i*1E9 if i is not None else i for i in [A,B,C]] # in Hz
    data["rotsigma"] = sigma
    data["islinear"] = linear
    data["freqs"   ] = wavenum_m # in 1/m
    data["ZPE"     ] = ZPE       # in hartree
    data["unpaired"] = unpaired
    data["E0"      ] = E0        # in hartree

    # return data
    return data

# ---- print information of interest after Hessian calc ----
def pyscf_printdata(data):

    # unpack data
    mass     = data["mass"    ]
    A,B,C    = data["rotcons" ]
    linear   = data["islinear"]
    sigma    = data["rotsigma"]
    freqs    = data["freqs"   ]
    ZPE      = data["ZPE"     ]
    unpaired = data["unpaired"]
    E0       = data["E0"      ]

    # print fata
    INFO  = rf"     - total mass (amu)              : {mass:.2f}" + "\n"
    if linear: INFO += rf"     - rotational constant  (GHz)    : {(A*1E-9):.2f}" + "\n"
    else     : INFO += rf"     - rotational constants (GHz)    : " + "  ".join(["%.2f"%(ii*1E-9) for ii in (A,B,C)]) + "\n"
    INFO += rf"     - rotational symmetry number    : {sigma:d}" + "\n"
    INFO += rf"     - vibrational wavenumbers (1/cm):"+"\n"
    # frequencies are, actually, wavenumbers in (1/m)
    for i in range(0,len(freqs),5):
        INFO += "         "+"  ".join(["%8.2f"%(ii/100) for ii in freqs[i:i+5]])+"\n"
    INFO += rf"     - zero point energy (hartree)   : {ZPE:.5f}"+"\n"
    print(INFO)

# ---- OPT + FREQ CALCULATION ----
def optimize_and_freqs(molecule,unpaired,charge,functional,basis,bsym=False):
    # files
    xyz_guess,xyz_opt,output_opt,output_frq = files_of_interest(molecule,functional,basis)
    # check if calculation was already carried out
    args = (molecule,unpaired,charge,functional,basis,bsym)
    # geometry optimization
    if not os.path.exists(xyz_opt): pyscf_carryout_opt(*args)
    # Hessian calculation
    mol, mf, hessian = pyscf_carryout_frq(*args)
    # Extract data
    return pyscf_extract(mol,mf,hessian,unpaired)

# ---- translational partition function ----
def pfn_translational(T,mass):

    beta    = 1/(k_B*T)

    # to SI
    mass_SI = mass * m_u

    # translational contribution
    V_per_molec = 1 / (beta * P_o)
    broglie_wvl = ((beta * h**2) / (2 * np.pi * mass_SI))**(0.5)
    q_translat  = V_per_molec / broglie_wvl**3

    # d(lnq_tr)/dbeta at constant volume
    dlnqdbeta_v = - 3/2 * (1/beta)

    # d(lnq_tr)/dbeta at constant pressure
    dlnqdbeta_p = - 5/2 * (1/beta)

    return q_translat, dlnqdbeta_v

# ---- rotational partition function ----
def pfn_rotational(T,A,B,C,linear,sigma):

    beta = 1/(k_B*T)

    # rotational constants (Hz --> 1/m) and rot temperature
    A /= c_0; theta_A = (h*c_0/k_B) * A

    if linear:
      q_rotational = T / theta_A
      dlnqdbeta    = - (1/beta)
    else:
      B /= c_0; theta_B = (h*c_0/k_B) * B
      C /= c_0; theta_C = (h*c_0/k_B) * C
      q_rotational = (np.pi * T**3 / (theta_A * theta_B * theta_C))**(1/2)
      dlnqdbeta    = - 3/2 * (1/beta)

    q_rotational /= sigma

    return q_rotational, dlnqdbeta

# ---- vibrational partition function ----
def pfn_vibrational(T,freqs):

    beta = 1/(k_B*T)

    q_vibrational = 1.0
    dlnqdbeta     = 0.0
    for freq in freqs:
        nu    = freq * c_0 # in Hz
        theta = h*nu/k_B
        q_vibrational *= 1 / (1 - np.exp(-theta/T)) # from ZPE
        dlnqdbeta     += - h*nu / (np.exp(theta/T) - 1)

    return q_vibrational, dlnqdbeta

# ---- electronic partition function ----
def pfn_electronic(T,unpaired):

    q_electronic = unpaired + 1
    dlnqdbeta    = 0.0

    return q_electronic, dlnqdbeta

# ---- compute U, H, S, and G ----
def compute_thermodynamics(T,molecule,key,thermodata):

    # unpack data
    natoms   = thermodata[molecule][key]["natoms"  ]
    mass     = thermodata[molecule][key]["mass"    ]
    A,B,C    = thermodata[molecule][key]["rotcons" ]
    linear   = thermodata[molecule][key]["islinear"]
    sigma    = thermodata[molecule][key]["rotsigma"]
    freqs    = thermodata[molecule][key]["freqs"   ]
    unpaired = thermodata[molecule][key]["unpaired"]
    E0       = thermodata[molecule][key]["E0"      ]
    ZPE      = thermodata[molecule][key]["ZPE"     ]

    # Get partition functions
    q_translat   , dlnqtdbeta = pfn_translational(T,mass)
    if natoms == 1:
       q_rotational , dlnqrdbeta = 1.0, 0.0
       q_vibrational, dlnqvdbeta = 1.0, 0.0
    else:
       q_rotational , dlnqrdbeta = pfn_rotational(T,A,B,C,linear,sigma)
       q_vibrational, dlnqvdbeta = pfn_vibrational(T,freqs)
    q_electronic , dlnqedbeta = pfn_electronic(T,unpaired)
    q_tot         = q_translat * q_rotational * q_vibrational * q_electronic
    dlnqdbeta_tot = dlnqtdbeta + dlnqrdbeta + dlnqvdbeta + dlnqedbeta

    # Info line
    line  = rf" {q_translat:.4E} | {q_rotational:.4E} | {q_vibrational:.4E} | {q_electronic:.4E} | {q_tot:.4E} | {(E0+ZPE):12.5f}"

    # calculate U and S (in S.I.; per molecule)
    Eref = (E0+ZPE)*Eh
    U    = Eref - dlnqdbeta_tot
    S    = - dlnqdbeta_tot/T + k_B * np.log(q_tot * np.e)

    # calculate H and G (per molecule)
    H    = U + k_B * T
    G    = H - S   * T

    # return data in J and J/K
    return U, H, S, G, line

# ---- Functions for fitting: best valur of Delta_{r}{C}_p^o ----
def sum_squared_errors(DCP,T,DGo_T,T_ref,DHo_ref,DSo_ref):
    refdata   = (DHo_ref,DSo_ref,DCP*R,T_ref)
    DGo_model = get_DGo(T,refdata)
    residuals = DGo_model - DGo_T
    return np.sum(residuals**2)

def calculate_rms(x1,x2):
    return (sum([(i-j)**2 for i,j in zip(x1,x2)])/len(x1))**0.5

#=====================================================================#
def freq_to_nu_and_theta(freq_cm):
    nu    = 100*freq_cm*c_0
    theta = h*nu/k_B
    return nu, theta
#---------------------------------------------------------------------#
def vib_contribution(freq_cm,T):
    nu,theta = freq_to_nu_and_theta(freq_cm)
    thetaT   = theta/T
    contri   = (thetaT * np.exp(-thetaT/2)/(1-np.exp(-thetaT))) **2
    return contri
#---------------------------------------------------------------------#
def vib_contri_avera(freq_cm,T1,T2):
    nu,theta = freq_to_nu_and_theta(freq_cm)
    average  = 1/(np.exp(theta/T2)-1) - 1/(np.exp(theta/T1)-1)
    average  = average * theta/(T2-T1)
    return average
#---------------------------------------------------------------------#
def plot_average(Tmin,Tmax,freqmin,freqmax,freq=None):
    freqs    = np.linspace(freqmin,freqmax,101)
    averages = vib_contri_avera(freqs,Tmin,Tmax)
    yy_inf   = vib_contribution(freqs,Tmin)
    yy_sup   = vib_contribution(freqs,Tmax)
    plt.plot(freqs,yy_inf  ,'r--',label=rf"$T$={Tmin:.2f} K",zorder=2)
    plt.plot(freqs,yy_sup  ,'b--',label=rf"$T$={Tmax:.2f} K")
    plt.plot(freqs,averages,'k-' ,label=rf"average")
    plt.xlabel(r"Vib. frequency (cm$^{-1}$)"     ,fontsize=14)
    plt.ylabel(r"Vib. contribution, $n^{V^\ast}$",fontsize=14)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    # --- update global variable: last_fig ---
    global last_fig
    last_fig = plt.gcf()

    if freq is not None:
       # limits so far
       xlim = plt.gca().get_xlim()
       ylim = plt.gca().get_ylim()
       # get average contribution for selected frequency
       aver =  vib_contri_avera(freq,Tmin,Tmax)
       # plot data for selected frequency
       plt.plot([freq,freq],[ylim[0],aver],'--',color="grey",zorder=1)
       plt.plot([xlim[0],freq],[aver,aver],'--',color="grey",zorder=1)
       plt.plot(freq,aver,'o',color="grey",label=rf"$n^{{V^\ast}} = {aver:.3f}$")
       # keep origonal limits
       plt.xlim(xlim)
       plt.ylim(ylim)
    plt.legend(loc="best",fontsize=11)

    # --- Show and close figure ---
    plt.show()
    plt.close()
#=====================================================================#


### **Loading the reaction**  


Execute the following cell in order to define your reaction.

In [ ]:
#@title <small><small> { display-mode: "form" }

NUS, MOLECULES = ask_for_reaction(max_nerr=3)


### **1. A Chemical Thermodynamics Perspective**  


####
We now need the thermodynamic parameters for this reaction (you may consult, for example, Atkins' *Physical Chemistry*, **2014**, Oxford University Press):

Execute the following cell and provide the required data when prompted.

In [ ]:
#@title <small><small> { display-mode: "form" }

magnitudes = []
magnitudes += [("H","formation entalphy [kJ/mol]")]
magnitudes += [("S","entropy [J/(mol K)]")]
magnitudes += [("C","molar heat capacities at constant P [J/(mol K)]")]

print(rf"We need the thermodynamical data for all the species.")

# ---- Get thermodynamical data ----
T_ref = ask_for_float("First, set the reference temperature (K): ")
print("")
thermodata = {}
for molecule in MOLECULES:
    print(rf"Introduce the thermodynamical data for species: {molecule:s}")
    print("")
    thermodata[molecule] = {}
    for key,magnitude in magnitudes:
        count = 0
        thermodata[molecule][key] = ask_for_float(rf"    * {magnitude:47s}: ")
    print("")

# ---- Get values for the reaction ----
DHo_ref  = sum([nu_i*thermodata[molec_i]["H"] for nu_i, molec_i in zip(NUS,MOLECULES)])
DSo_ref  = sum([nu_i*thermodata[molec_i]["S"] for nu_i, molec_i in zip(NUS,MOLECULES)])
DCPo_ref = sum([nu_i*thermodata[molec_i]["C"] for nu_i, molec_i in zip(NUS,MOLECULES)])

# ---- Print values for the reaction ----
print(rf"Magnitudes of interest at {T_ref:.2f} K:")
print(rf"    Delta_r{{H}}^o  = {DHo_ref:+8.2f} kJ/mol")
print(rf"    Delta_r{{S}}^o  = {DSo_ref:+8.2f}  J/(mol K)")
print(rf"    Delta_r{{Cp}}^o = {DCPo_ref:+8.2f}  J/(mol K)")

# ---- Convert Entalphy of reaction to SI ----
DHo_ref *= 1000

# ---- Save data ----
REFDATA = (DHo_ref,DSo_ref,DCPo_ref,T_ref)
#------------------------------------------------------

####
(a) EFFECT OF THE TEMPERATURE ON $\Delta_r G^\circ$

#####
For an ideal gas-phase reaction, the temperature dependence of the standard Gibbs free energy change, $\Delta_{r} G^\circ$, can be expressed as:

$$
\Delta_{r} G^{\circ}(T)= \Delta_{r} H^{\circ}(T_{\rm ref})-T \cdot \Delta_{r} S^{\circ}(T_{\rm ref})+\Delta_{r} C_p^\circ \cdot \left[ T-T_{\rm ref}+T \cdot \ln \left(\frac{T_{\rm ref}}{T}\right)\right]
 \tag{1}
$$

with $T_{\rm ref}$ being the reference $T$ and under the assumption that $\Delta_{r} C_p^\circ$ is independent of $T$. Notice that the corresponding equilibrium constant is then obtained as:

$$
K_p^\circ(T) = e^{-\Delta_{r} G^{\circ}(T)/(R \cdot T)}
 \tag{2}
$$

Examine the temperature dependence of $\Delta G^\circ$ (and of $K_p$) by executing the next cell. In the generated plots, the red dot indicates the value at $T_{\rm ref}$.

In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------
T     = np.linspace(200,400,31)
DGo_T = plot_DG_T(T,T_ref,REFDATA)
#------------------------------------------------------

#####
According to the Gibbs–Helmholtz equation, the following equality should hold:

$$
\frac{\Delta_{r} H^{\circ}}{T^2} = \frac{d}{dT}\left( -\frac{\Delta_{r} G^{\circ}}{T} \right)
\tag{3}
$$

This relationship can be tested by computing the numerical derivative of the term on the right-hand side and comparing it with the analytical expression on the left-hand side.

- The analytical expression for the enthalpy term as a function of temperature is:
$$
\Delta_{r} H^{\circ} = \Delta_{r} H^{\circ}(T_{\rm ref}) + \Delta_{r} C_{p}^\circ \cdot (T- T_{\rm ref})
\tag{4}
$$
Using this expression, we can easily plot $\Delta_{r} H^{\circ}/ T^2$.

- On the other side, the numerical derivative of $-\Delta_{r} G^{\circ}/T$ with respect to temperature can be directly computed from the data obtained in the execution of the previous cell.

Execute the next cell to check the Gibbs–Helmholtz relationship.

In [ ]:
#@title <small><small> { display-mode: "form" }

plot_gibbshelmholtz(T,DGo_T,REFDATA)

####
(b) REACTION CONDUCTED AT CONSTANT _P_ AND _T_

#####
$G$ changes dynamically as the reaction progresses. By knowing the initial conditions for the reaction, it is possible to track how $G$ varies with the extent of reaction ($\xi$), which is of special interest when the reaction takes place at constant $T$ and $p$.

Firstly, set the initial number of moles of each species involved in your reaction by executing the next cell.

In [ ]:
#@title <small><small> { display-mode: "form" }

print("Let us set the initial conditions")
print("")

print("   - indicate, first, the number of moles of each species!")
#-----------------------------------
n_0 = []
for molecule in MOLECULES:
    nmol = ask_for_float(rf"     n_mol({molecule:s}): ")
    n_0.append(nmol)
n_0 = np.array(n_0)
#-----------------------------------
STEP          = 1E-4
xi_min,xi_max = limits_xi(n_0,NUS)
xis           = np.arange(xi_min,xi_max+STEP,STEP)
fixed_args    = (MOLECULES,NUS,n_0,xis,REFDATA)
#-----------------------------------
print("")
print(rf"   min value for xi: {xi_min:+7.3f} mol")
print(rf"   max value for xi: {xi_max:+7.3f} mol")
print("")

Now, execute the cell below, set the temperature and the total pressure, and analyze how $G$ changes with $\xi$:

In [ ]:
#@title <small><small> { display-mode: "form" }

#----  Get limit values for xi  ----
STEP          = 1E-4
xi_min,xi_max = limits_xi(n_0,NUS)
#----     Get values for xi     ----
xis           = np.arange(xi_min,xi_max+STEP,STEP)
#----   Save info in a tuple    ----
fixed_args    = (MOLECULES,NUS,n_0,xis,REFDATA)
#-----------------------------------

# Enable Colab’s custom widget manager, allowing interactive ipywidgets to function correctly
output.enable_custom_widget_manager()

# -------- Sliders --------
args     = dict(layout=w.Layout(width='600px'),style={'description_width': '150px'},continuous_update=True,readout_format='.2f')
T_slider = w.FloatSlider(value=298.00,min=200.00,max=400.00,step=1.00,description=r'T [K]'  , **args)
P_slider = w.FloatSlider(value=  1.00,min=  0.01,max=  3.00,step=0.01,description=r'p [bar]', **args)
ui       = w.VBox([T_slider,P_slider])

# -------- download button --------
btn      = w.Button(description='Download current figure', icon='download', button_style='primary',layout=w.Layout(width='200px', height='30px'))
btn.on_click(lambda b: _on_download_clicked(b,"PT"))

# -------- slider ---> function --------
# notice that P is converted from bar to Pa with *1E5
out = w.interactive_output(lambda T, P, fixed_args: plot_DG_TP(T, P*1E5, fixed_args), {'T': T_slider, 'P': P_slider, 'fixed_args': w.fixed(fixed_args)})
display(w.VBox([ui, btn]), out)


####
(c) REACTION CONDUCTED AT CONSTANT _V_ AND _T_

#####
Another case of interest is a reaction carried out in a reactor at constant $T$ and $V$. Under these conditions, the relevant thermodynamic potential is the Helmholtz free energy (A):

$$
A = G - pV = G - nRT
\tag{7}
$$

Let us assume that the reaction vessel initially contains the number of moles that you defined previously. Execute the cell below, set the temperature and the total volume, and analyze how $A$ changes with $\xi$:


In [ ]:
#@title <small><small> { display-mode: "form" }

#----  Get limit values for xi  ----
STEP          = 1E-4
xi_min,xi_max = limits_xi(n_0,NUS)
#----     Get values for xi     ----
xis           = np.arange(xi_min,xi_max+STEP,STEP)
#----   Save info in a tuple    ----
fixed_args    = (MOLECULES,NUS,n_0,xis,REFDATA)
#-----------------------------------

# Enable Colab’s custom widget manager, allowing interactive ipywidgets to function correctly
output.enable_custom_widget_manager()

# -------- Sliders --------
args     = dict(layout=w.Layout(width='600px'),style={'description_width': '150px'},continuous_update=True,readout_format='.2f')
T_slider = w.FloatSlider(value=298.00,min=200.00,max=400.00,step=1.00,description=r'T [K]', **args)
V_slider = w.FloatSlider(value= 24.78,min=  2.00,max=250.00,step=0.01,description=r'V [L]', **args)
ui       = w.VBox([T_slider,V_slider])

# -------- download button --------
btn      = w.Button(description='Download current figure', icon='download', button_style='primary',layout=w.Layout(width='200px', height='30px'))
btn.on_click(lambda b: _on_download_clicked(b,"VT"))

# -------- slider ---> function --------
# V is converted from L to m3 with *1E-3
out = w.interactive_output(lambda T, V, fixed_args: plot_DA_TV(T, V*1E-3, fixed_args), {'T': T_slider, 'V': V_slider, 'fixed_args': w.fixed(fixed_args)})
display(w.VBox([ui, btn]), out)

### **2. A Statistical-Mechanical Perspective**  


#####
In the previous section, we analyzed the equilibrium using tabulated values of $\Delta_{r} H^{\circ}(T_{\rm ref})$, $\Delta_{r} S^{\circ}(T_{\rm ref})$ and $\Delta_{r} C_p^\circ(T_{\rm ref})$, which were then inserted into a macroscopic expression for $\Delta_{r} G^{\circ}(T)$. In this section, we will obtain these thermodynamic quantities from a molecular-level description using statistical mechanics.

For an ideal-gas mixture, the standard thermodynamic functions of each species can be derived from its molecular partition function, $q^\circ(T)$, which factorizes into translational, rotational, vibrational, and electronic contributions. From $q^\circ(T)$ we can compute $G^{\circ}_{m}(T)$, $H^{\circ}_{m}(T)$, $S^{\circ}_{m}(T)$, and $C^{\circ}_{p,m}(T)$ for each molecule. The reaction quantities then follow by applying the stoichiometric sum over reactants and products. In this way, the temperature dependence of
$\Delta_{r} G^{\circ}(T)$ is linked directly to molecular structure and vibrational spectra.

To start, we need initial molecular geometries for reactant(s) and product(s). These structures will be used as input for subsequent electronic-structure calculations (geometry optimization and harmonic vibrational analysis), from which we will extract vibrational frequencies and other molecular parameters required to build the partition functions.

#### _(a) Guess geometries of reactants and products_

This notebook can either retrieve the geometry of a species from PubChem[[🌐]](https://pubchem.ncbi.nlm.nih.gov/) by using its PubChem CID identifier[[🌐]](https://pubchem.ncbi.nlm.nih.gov/compound/25352); or it can build the molecule from its SMILES representation[[🌐]](https://en.wikipedia.org/wiki/Simplified_Molecular_Input_Line_Entry_System) using with the **Rdkit** toolkit[[🌐]](https://www.rdkit.org/).

Run the following cell and enter the PubChem CID or the SMILES code for each species.
Once the geometry is retrieved/generated, its **3D geometry** will be displayed in an interactive viewer that allows you to rotate and inspect the molecule freely.


In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------
MOLECULES_ID  = {k:None for k in MOLECULES}
#------------------------------------------------------

print("Getting structures. Insert one of the following:")
print("   - its SMILES code        [e.g. CCO]")
print("   - its PubChem CID number [e.g. 702]")
print("")

# Ask for CID or SMILES
for molecule in MOLECULES:
    MOLECULES_ID[molecule] = input(rf"   * {molecule:s}: ").strip()
print("")

# Get structures
for molecule in MOLECULES:
    print(rf"   * Obtaining guess geometry for {molecule:s}")
    mol_id    = MOLECULES_ID[molecule]
    xyz_guess = files_of_interest(molecule)[0]
    for getguess in [pubchem_cid,rdkit_smiles2geom]:
        symbols,coords,smiles = getguess(mol_id)
        if symbols is not None: break
    if symbols is None: continue
    # write file
    data_2_xyz(symbols,coords,xyz_guess,smiles)
    # visualize
    view = create_visualization_xyz(xyz_guess)
    view.show()
    print("")


#####
The molecular structures obtained so far are **not** optimized; they do not correspond to the **minimum-energy geometries**.

Before performing any optimization, however, we must specify for each molecule both its total charge and the number of unpaired electrons.

Run the next cell and enter the value for these variables:

In [ ]:
#@title <small><small> { display-mode: "form" }

CHARGES,UNPAIREDS = {},{}
print("Insert the total charge (a.u.) and number of unpaired electrons for each molecule (integers only):")
print("")
for molecule in MOLECULES:
    print(rf"   * {molecule:s}")
    charge   = input("     total molecular charge [in au] = ").strip()
    unpaired = input("     number of unpaired electrons   = ").strip()
    CHARGES[molecule]   = int(charge)
    UNPAIREDS[molecule] = int(unpaired)
    print("")

#### _(b) DFT calculations_

#####

Now, we are ready to perform the quantum-chemistry calculations using **PySCF**[[🌐]](https://pyscf.org). After optimizing the geometries, we will also compute the **Hessian matrix** to verify that each structure corresponds to a true minimum (all vibrational frequencies are real), rather than a different type of stationary point. Moreover, these vibrational frequencies are required to obtain the vibrational partition function, which is essential for evaluating thermodynamic quantities.

Run the following cell to enter a DFT functional and basis set. Make sure that both are supported by PySCF. Once inserted, the quantum-chemistry calculations will be performed.

**Note:** The DFT integration grid was set to **4** though the `DFTGRID` variable (0 - 9). Modify it in the cell if needed (big number for large mesh grids).



In [ ]:
#@title <small><small> { display-mode: "form" }

# =====================================================
# THIS VARIABLE CAN BE MODIFIED:
DFTGRID = 4
# =====================================================

#------------------------------------------------------
if "THERMODATA" not in globals(): THERMODATA = {k:{}   for k in MOLECULES}
if "KEYS"       not in globals(): KEYS       = []
#------------------------------------------------------
# Ask for functional and basis
functional = input("  insert DFT functional: ").strip()
basis      = input("  insert basis set     : ").strip()
print("")
#------------------------------------------------------
# Print information
key = (functional,basis)
print(rf" - Functional: {key[0].upper():s}")
print(rf" - Basis set : {key[1].upper():s}")
print(rf" - Grid level: {DFTGRID:d}")
print("")
#------------------------------------------------------
# Carry on calculations
t1 = time.time()
for molecule in MOLECULES:
    print(rf" * Molecule: {molecule:s}")
    THERMODATA[molecule][key] = optimize_and_freqs(molecule,UNPAIREDS[molecule],CHARGES[molecule],key[0],key[1])
    print("")
    pyscf_printdata(THERMODATA[molecule][key])
    pyscf_download(molecule,key[0],key[1],[1])
    print("")
    # visualization
    xyz_opt = files_of_interest(molecule,key[0],key[1])[1]
    view    = create_visualization_xyz(xyz_opt)
    view.show()
    print("")
t2 = time.time()
print(" - total elapsed time: %.2f minutes"%((t2-t1)/60))
#------------------------------------------------------
# save successful key
if key not in KEYS: KEYS.append(key)
#------------------------------------------------------

#####

PySCF does not always return the correct symmetry number ($\sigma$) due to numerical inaccuracies [[🌐]](https://link.springer.com/article/10.1007/s00214-007-0328-0). Therefore, before moving to (c) and compute the partition functions, we must ensure that the symmetry numbers are correct.

Verify these values in the preceding optimization cells, and run the following cell if any symmetry numbers need to be corrected.

In [ ]:
#@title <small>💻 Correct symmetry numbers <small> { display-mode: "form" }
print("Insert the rotational symmetry number for each molecule:")
print("")
for molecule in MOLECULES:
    sigma = int(input(rf"   * sigma({molecule:s}) = ").strip())
    print("")
    for functional,basis in KEYS:
        key = (functional,basis)
        try:
          sigma_old = THERMODATA[molecule][key]["rotsigma"]
          print(rf"     {functional.upper():8s}: {sigma_old:d} --> {sigma:d}")
          THERMODATA[molecule][key]["rotsigma"] = sigma
        except:
          print(rf"     data not found for {functional.upper():s}/{basis.upper():s}...")
    print("")

#### _(c) Calculation of partition functions and thermodynamic functions_


#####
We can now evaluate the partition functions of all species participating in the reaction at 298.15 K. From them, the thermodynamic state functions can be determined.

In [ ]:
#@title <small><small> { display-mode: "form" }

TABLE  = " ----------------------------------------------------------------------------------------------" + "\n"
TABLE += "  FUNCTIONAL/BASIS_SET | DELTA_r{U}^o | DELTA_r{H}^o | DELTA_r{S}^o | DELTA_r{G}^o |   K_p^o   " + "\n"
TABLE += " ----------------------|--------------|--------------|--------------|--------------|-----------" + "\n"

print("Partition functions and zero-point–corrected total energies (E0 + ZPE); energies in hartree.")
print("")

REFDATADFT = {}
for key in KEYS:
    level = rf"{key[0].upper():8s}/{key[1].upper():s}"
    allok = True

    SUBTABLE  = "    --------------------------------------------------------------------------------------------"+"\n"
    SUBTABLE += "      molecule |   pfn tr    |  pfn rot   |  pfn vib   |  pfn ele   |  pfn tot   |   E0 + ZPE   "+"\n"
    SUBTABLE += "    -----------|-------------|------------|------------|------------|---------------------------"+"\n"

    DUo,DHo,DSo,DGo  = 0.,0.,0.,0.
    for nu_i,molecule in zip(NUS,MOLECULES):

        output_frq = files_of_interest(molecule,key[0],key[1])[3]
        if not os.path.exists(output_frq):
           allok = False
           break
        # calculate thermodynamics
        try: Uo, Ho, So, Go, line = compute_thermodynamics(T_ref,molecule,key,THERMODATA)
        except: allok = False; break
        SUBTABLE += rf"      {molecule:8s} | " + line + "\n"

        # reaction magnitudes
        DUo += nu_i * Uo
        DHo += nu_i * Ho
        DSo += nu_i * So
        DGo += nu_i * Go
    SUBTABLE += "    --------------------------------------------------------------------------------------------"+"\n"
    if not allok: continue
    print(rf"    ==> {level:s} <==")
    print("")
    print(SUBTABLE)
    # Equilibrium constant
    Kp = np.exp(-DGo/k_B/T_ref)
    # Print info
    if Kp > 1E4: ff = "9.2E"
    else       : ff = "9.3f"
    TABLE += rf"  {level:20s} | {DUo*NA/1000:12.2f} | {DHo*NA/1000:12.2f} | {DSo*NA:12.2f} | {DGo*NA/1000:12.2f} | {Kp:{ff:s}} " + "\n"
    # save reaction magnitudes
    REFDATADFT[key] = T_ref,DUo*NA,DHo*NA,DSo*NA,DGo*NA
TABLE += " ----------------------------------------------------------------------------------------------" + "\n"
TABLE += "\n"
TABLE += "     units of DELTA_r{U}^o ==> kJ/mol"   + "\n"
TABLE += "     units of DELTA_r{H}^o ==> kJ/mol"   + "\n"
TABLE += "     units of DELTA_r{S}^o ==>  J/mol/K" + "\n"
TABLE += "     units of DELTA_r{G}^o ==> kJ/mol"   + "\n"

print("")
print(TABLE)
print("")

#### _(d) Temperature dependence of $\Delta_{r}G^\circ$: $\Delta_{r}C_{p}^\circ$ and the vibrational degrees of freedom_


#####

The calculation of $\Delta_{r}G^\circ$ in (c) can be performed at temperatures other than $T_{\rm ref}$. As you may recall from the first part of this Notebook, this temperature dependence should behaves:

$$
\Delta_{r} G^{\circ}(T) = \Delta_{r} H^{\circ}(T_{\rm ref})-T \cdot \Delta_{r} S^{\circ}(T_{\rm ref})+\Delta_{r} C_p^\circ \cdot \left[ T-T_{\rm ref}+T \cdot \ln \left(\frac{T_{\rm ref}}{T}\right)\right]
\tag{1}
$$

under the assumption that $\Delta_{r}C_{p}^\circ$ is constant. Note that you already calculated all quantities at the reference temperature ($T_{\rm ref}$), except for $\Delta_{r}C_{p}^\circ$ itself.

Next, we will obtain the value that best reproduces the numerical results obtained for $\Delta_{r}G^\circ$.

#####

The optimal value of $\Delta_{r} C_p^\circ$ can be obtained directly by fitting the computed $\Delta_{r} G^\circ(T)$ values (calculated using Statistical Thermodynamics) to Eq. (1). In other words, instead of choosing  $\Delta_{r} C_p^\circ$ manually, one could determine the value that minimizes the deviation between the analytical expression, Eq. (1), and the data, yielding the best agreement across the temperature range.

To do this, run the following cell.

In [ ]:
#@title <small><small> { display-mode: "form" }

# Temperature dependence
Tmin  = max(T_ref-150,0)
Tmax  = T_ref+150
T     = np.linspace(Tmin,Tmax,10)
for key_i in THERMODATA[MOLECULES[0]].keys():
    # Print level of calculation
    level = rf"{key_i[0].upper():8s}/{key_i[1].upper():s}"
    print(rf"==> {level:s} <==" + "\n")
    # reference data
    T_ref, DUo_ref, DHo_ref, DSo_ref, DGo_ref = REFDATADFT[key_i]
    # Calculate D_{r}G^o at different temperatures using Statistical Thermodynamics
    print(rf"    Calculating Delta_r{{G}}^o between {Tmin:.2f} and {Tmax:.2f} K using Statistical Thermodynamics...")
    DGo_T = []
    for Ti in list(T):
        DGo_i = [nu_i * compute_thermodynamics(Ti,molecule,key_i,THERMODATA)[3] for nu_i,molecule in zip(NUS,MOLECULES)]
        DGo_T.append(float(sum(DGo_i)*NA))
    DGo_T = np.array(DGo_T)
    print("")

    # minimize
    print(rf"     - fitting to Eq. (1) using values at T_ref = {T_ref:.2f} K...")
    res       = minimize(sum_squared_errors,np.array([0.0]),args=(T,DGo_T,T_ref,DHo_ref,DSo_ref))
    dof_best  = res.x[0]

    # Values at optimized results (and other values)
    refdata   = (DHo_ref,DSo_ref,dof_best*R,T_ref)
    DGo_model = get_DGo(T,refdata)
    RMS       = calculate_rms(DGo_T,DGo_model)/1000

    print(rf"     - best fit obtained for Delta_{{r}}C_{{p}}^o = {dof_best:.2f} * R   [RMS = {RMS:.3f} kJ/mol]")
    print("")

    # Plotting
    plot_DG_T_2(T,DGo_T,DGo_model,dof_best*R,T_ref,DGo_ref,key_i)
    print()

    print("    -------------------------------------------")
    print("     Molecule | rot dof  | vib dof  |  C_{p,m} ")
    print("    -------------------------------------------")
    DCP = 0.0
    for i,molecule in enumerate(MOLECULES):
        # check if data is available
        if molecule not in THERMODATA: continue
        if key_i    not in THERMODATA[molecule]: continue
        # rotational dofs
        if THERMODATA[molecule][key_i]["islinear"]: rdof = 2
        else                                      : rdof = 3
        # calculate vdof and Cp
        freqs_cm = [ii/100 for ii in THERMODATA[molecule][key_i]["freqs"]]
        vdof     = sum([vib_contri_avera(freq_cm,Tmin,Tmax) for freq_cm in freqs_cm])
        Cp       = R + (3+rdof+2*vdof)/2*R
        DCP     += Cp*NUS[i]
        print(rf"     {molecule:7s}  |  {rdof:6d}  |  {vdof:6.3f}  | {Cp/R:6.3f}*R  ")
    print("    -------------------------------------------")
    print(rf"                 ==> Delta_r{{C_p}}^o = {DCP/R:6.2f}*R")
    print("")
